In [14]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import requests


# LOAD DATA & EXTRACT 80 SAMPLES FOR GPT

In [20]:
print("Loading data and extracting the 80-sample test set...")
df = pd.read_csv('./resources/cn7050data.csv', encoding='latin-1', names=['sentiment', 'text'])


print("before removing duplicates:", df.shape)
print(df[df.duplicated(subset=["sentiment","text"],keep=False)])
df = df.drop_duplicates()
df=df.reset_index(drop=True)
df = df.drop_duplicates().dropna()
print("after removing duplicates:", df.shape)
df['sentiment'] = df['sentiment'].str.lower()
#df[df.duplicated(subset=["sentiment","text"],keep=False)]

#df.tail(7)


Loading data and extracting the 80-sample test set...
before removing duplicates: (4846, 2)
     sentiment                                               text
1098   neutral  The issuer is solely responsible for the conte...
1099   neutral  The issuer is solely responsible for the conte...
1415   neutral  The report profiles 614 companies including ma...
1416   neutral  The report profiles 614 companies including ma...
2395   neutral  Ahlstrom 's share is quoted on the NASDAQ OMX ...
2396   neutral  Ahlstrom 's share is quoted on the NASDAQ OMX ...
2566   neutral  SSH Communications Security Corporation is hea...
2567   neutral  SSH Communications Security Corporation is hea...
3093   neutral  Proha Plc ( Euronext :7327 ) announced today (...
3094   neutral  Proha Plc ( Euronext :7327 ) announced today (...
3205   neutral  The company serves customers in various indust...
3206   neutral  The company serves customers in various indust...
after removing duplicates: (4840, 2)


In [24]:
OLLAMA_URL = "http://localhost:11434/api/generate" 
MODEL_NAME = "gpt-oss:20b-cloud"

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['sentiment'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['sentiment'])

gpt_test_df = test_df.sample(n=80, random_state=13).copy()
print(f"Successfully sampled {len(gpt_test_df)} headlines for GPT-OSS evaluation.\n")



def build_reasoning_prompt(headline):
    
    prompt = f"""You are an expert financial analyst. Read the following financial news headline and determine its sentiment.
    Headline: "{headline}"

Step 1: Analyze the financial implications of this headline. (Is it about growth, loss, standard operations, etc.?)
Step 2: Based on your analysis, decide if the sentiment is strictly Positive, Negative, or Neutral.

Provide your output in exactly this format:
\nReasoning: [...]
\nFinal Label: [positive / negative / neutral]
"""
    return prompt

def call_moodle_gpt_oss_model(prompt):
    return "Reasoning: The company shows growth and strategic alignment. \nFinal Label: positive"


print("Running GPT-OSS and extracting reasoning...")

def call_ollama(prompt):
    payload={
        "model":MODEL_NAME,
        "prompt":prompt,
        "stream":False,
    }
    try:
        response=requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()
        return response.json().get("response", "")
    except requests.exceptions.RequestException as e:
        print(f"Error calling Ollama API: {e}")
        return "Reasoning: Error occurred while calling Ollama API. \nFinal Label: neutral"

reasonings = []
gpt_predictions = []

for index,row in gpt_test_df.iterrows():
    headline = row['text']
    prompt = build_reasoning_prompt(headline)
    
    response_text=call_ollama(prompt)
    reasoning_match = re.search(r"Reasoning:\s*(.*?)\s*Final Label:", response_text, re.DOTALL)
    label_match = re.search(r"Final Label:\s*(\w+)", response_text, re.IGNORECASE)
    
    reasoning = reasoning_match.group(1).strip() if reasoning_match else "No reasoning found"
    label = label_match.group(1).strip().lower() if label_match else "neutral" 
    reasonings.append(reasoning)
    gpt_predictions.append(label)
    
gpt_test_df['gpt_reasoning'] = reasonings
gpt_test_df['gpt_prediction'] = gpt_predictions

Successfully sampled 80 headlines for GPT-OSS evaluation.

Running GPT-OSS Inference and extracting reasoning... (This might take a few minutes)


In [ ]:
true_labels = gpt_test_df['sentiment'].tolist()

print("\n" + "="*40)
print(" GPT-OSS (20B) PERFORMANCE METRICS ")
print("="*40)
print(f"Accuracy: {accuracy_score(true_labels, gpt_predictions):.4f}")
print(f"Macro-F1 Score: {f1_score(true_labels, gpt_predictions, average='macro'):.4f}")

print("\n--- Detailed Per-Class Report ---")
print(classification_report(true_labels, gpt_predictions, target_names=['negative', 'neutral', 'positive']))

cm = confusion_matrix(true_labels, gpt_predictions)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('GPT-OSS Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: GPT-OSS (Reasoning)')
plt.show()